# Model Evaluation — Production YOLOv8n on Held-Out Test Set

**Project:** NazarBaan ANPR  
**Model:** `models/merged_yolov8n/train/weights/best.pt`  
**Evaluation set:** 65 held-out test images from the merged dataset (Burhan Khan + ubaidp1049, perceptual-hash deduplicated).

## What this notebook does

Phases 3 and 4 produced single-number metrics (mAP, precision, recall). Those are useful at a glance but they hide *where* and *how* the model fails — the information that matters most for deployment.

This notebook unpacks the test-set evaluation into:

1. **Per-image predictions** — every detection logged with confidence and IoU to its nearest ground-truth box.
2. **TP / FP / FN classification** — every prediction explicitly labeled as a true positive, false positive, or false negative.
3. **Failure-case gallery** — the actual images the model got wrong, visualized side-by-side with their labels.
4. **Confidence-threshold sweep** — precision and recall as functions of the detection confidence threshold. This tells us what threshold to ship in the gate app.
5. **CPU inference speed** — measured on this laptop. The deployment-feasibility number.

## Why this matters for the report

The brief asks for *"performance metrics and analysis"* and *"discussion of challenges and improvements."* A model that scores mAP 0.97 but fails on dirty plates is a fundamentally different product from one that scores 0.95 but fails uniformly. This notebook surfaces *which* one we have.

In [ ]:
"""Setup for local inference evaluation. CPU-only — same machine the gate
app will eventually run on, so timing measurements are representative."""

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from tqdm import tqdm
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
WEIGHTS = PROJECT_ROOT / "models" / "merged_yolov8n" / "train" / "weights" / "best.pt"
TEST_IMGS = PROJECT_ROOT / "data" / "processed" / "merged_v1" / "test" / "images"
TEST_LBLS = PROJECT_ROOT / "data" / "processed" / "merged_v1" / "test" / "labels"
FIG_DIR = PROJECT_ROOT / "reports" / "figures"

# Inference parameters — same as training imgsz so we're evaluating apples to apples
IMG_SIZE = 960
LOW_CONF = 0.001  # Run very low so I can later sweep across thresholds offline

print(f"Weights:     {WEIGHTS.name}  ({WEIGHTS.stat().st_size / 1e6:.1f} MB)")
print(f"Test images: {len(list(TEST_IMGS.glob('*.jpg')))}")
print(f"Inference imgsz: {IMG_SIZE}")

## 1. Per-Image Predictions vs Ground Truth

Running the model at a very low confidence threshold (0.001) so I capture every prediction the model would consider plausible. The threshold sweep later (Section 4) re-applies cutoffs offline using these raw predictions — much faster than re-running inference at every threshold.

For each image I'll:
- Run the model and collect every (xc, yc, w, h, conf) prediction.
- Load the ground-truth boxes from the label file.
- For each prediction, compute IoU against every ground-truth box and record the maximum.
- For each ground-truth box, record whether *any* prediction at IoU ≥ 0.5 matched it (used to count FN later).

Result: one DataFrame row per prediction, plus a separate set tracking which ground truths were ever matched. This is the data structure every downstream analysis pulls from.

In [ ]:
"""Run the production model on every test image and capture predictions + IoU
to the nearest ground-truth box."""

def load_yolo_labels(label_path: Path):
    """Return list of (class_id, xc, yc, w, h) normalized tuples."""
    if not label_path.exists():
        return []
    rows = []
    for line in label_path.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) == 5:
            rows.append((int(parts[0]), *map(float, parts[1:])))
    return rows


def iou_xyxy(boxA, boxB):
    """Standard Intersection-over-Union of two boxes in (x1, y1, x2, y2) format."""
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0


def yolo_to_xyxy_px(xc, yc, w, h, img_w, img_h):
    """Normalized YOLO -> pixel xyxy."""
    return (
        (xc - w/2) * img_w, (yc - h/2) * img_h,
        (xc + w/2) * img_w, (yc + h/2) * img_h,
    )


# Load model once — reused for every inference
model = YOLO(str(WEIGHTS))

predictions = []        # one row per predicted box
gt_matched  = []        # one row per ground-truth box; tracks if it was ever matched

test_images = sorted(TEST_IMGS.glob("*.jpg"))
print(f"Running inference on {len(test_images)} images...\n")

for img_path in tqdm(test_images):
    img = Image.open(img_path)
    img_w, img_h = img.size

    # Ground-truth boxes for this image
    gts_norm = load_yolo_labels(TEST_LBLS / f"{img_path.stem}.txt")
    gts_px = [yolo_to_xyxy_px(xc, yc, w, h, img_w, img_h) for _, xc, yc, w, h in gts_norm]

    # Inference at very low conf so I capture everything
    result = model(img_path, imgsz=IMG_SIZE, conf=LOW_CONF, verbose=False)[0]

    preds = result.boxes
    matched_gt_indices = set()

    for i in range(len(preds)):
        pred_xyxy = preds.xyxy[i].cpu().numpy().tolist()
        conf = float(preds.conf[i])

        # Find best-matching ground-truth box for this prediction
        ious = [iou_xyxy(pred_xyxy, gt) for gt in gts_px]
        best_iou = max(ious) if ious else 0.0
        best_gt_idx = ious.index(best_iou) if ious else -1

        if best_iou >= 0.5:
            matched_gt_indices.add(best_gt_idx)

        predictions.append({
            "image": img_path.name,
            "source": img_path.stem.split("__")[0],
            "conf": conf,
            "best_iou": best_iou,
        })

    # Track which ground-truth boxes never got matched at IoU >= 0.5 (top conf 1.0)
    for i in range(len(gts_px)):
        gt_matched.append({
            "image": img_path.name,
            "source": img_path.stem.split("__")[0],
            "gt_idx": i,
            "matched_at_iou50": i in matched_gt_indices,
        })

pred_df = pd.DataFrame(predictions)
gt_df   = pd.DataFrame(gt_matched)

print(f"\nTotal predictions captured: {len(pred_df)}")
print(f"Total ground-truth boxes:   {len(gt_df)}")
print(f"Ground truths matched @ IoU≥0.5: {gt_df['matched_at_iou50'].sum()} / {len(gt_df)}")